# Exploratory Analysis of Company Records

## Purpose

This notebook explores the structure and contents of the company dataset supplied for the OneCredit Data Scientist take-home assessment.

The objectives are to:

- Understand the available company information.
- Identify missing data and unusual records.
- Understand the financial, ownership and litigation fields.
- Identify information that may be useful when vetting suppliers or project partners.
- Select representative companies for developing and evaluating a summarisation system.

This notebook is for data exploration only. It does not generate or evaluate company summaries.

In [1]:
import json
from pathlib import Path
from pprint import pprint

In [4]:
Path.cwd()

WindowsPath('C:/Users/semee/Desktop/OC/oc-assessment/notebook')

In [5]:
data_path = Path("../data/companies.jsonl")

print(data_path.exists())
print(data_path)

True
C:\Users\semee\Desktop\OC\oc-assessment\data\companies.jsonl


## Load JSONL Records

In [6]:
with data_path.open(encoding="utf-8") as file:
    companies = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

In [7]:
len(companies)

50

## High Level Overview

In [16]:
company = companies[0]

In [17]:
company.keys()

dict_keys(['registrationNumber', 'profile', 'relationships', 'financials', 'litigation'])

In [18]:
pprint(company["profile"], sort_dicts=False)

{'companyName': 'Unity Global Trading Sdn. Bhd.',
 'status': 'Active',
 'companyType': 'Private limited company',
 'incorporationDate': '2018-06-15',
 'industry': 'Retail',
 'state': 'Selangor',
 'natureOfBusiness': 'Retail of textiles and baju kurung',
 'shareCapital': 250000,
 'formerNames': []}


In [19]:
print("Registration:", company["registrationNumber"])
print("Company:", company["profile"]["companyName"])
print("Directors:", len(company["relationships"]["directors"]))
print("Shareholders:", len(company["relationships"]["shareholders"]))
print("Financial years:", len(company["financials"]["years"]))
print("Court cases:", len(company["litigation"]["cases"]))

Registration: SYN-000029
Company: Unity Global Trading Sdn. Bhd.
Directors: 5
Shareholders: 4
Financial years: 3
Court cases: 0


## Drill into financial part

In [20]:

company_with_financials = next(          #search the companies list, and return the first record where the 'years' is not empty
    company
    for company in companies
    if company["financials"]["years"]
)

print(company_with_financials["profile"]["companyName"])
print(len(company_with_financials["financials"]["years"]))

Unity Global Trading Sdn. Bhd.
3


In [21]:
financial_years = company_with_financials["financials"]["years"]

for year in financial_years:
    print(year["financialYear"], year["financialYearEnd"])

2022 2022-06-30
2023 2023-06-30
2024 2024-06-30


In [22]:
latest_year = max(
    financial_years,
    key=lambda year: year["financialYear"]
)

print(latest_year.keys())
print(latest_year["audit"])
print(latest_year["statements"].keys())
print(latest_year["ratiosAndMetrics"].keys())

dict_keys(['financialYear', 'financialYearEnd', 'audit', 'statements', 'ratiosAndMetrics'])
{'firm': 'Audit Firm 8', 'opinion': 'Unqualified'}
dict_keys(['revenue', 'costOfSales', 'grossProfit', 'operatingExpenses', 'operatingProfit', 'depreciation', 'interestExpense', 'profitBeforeTax', 'taxExpense', 'profitAfterTax', 'cash', 'tradeReceivables', 'inventory', 'currentAssets', 'nonCurrentAssets', 'totalAssets', 'tradePayables', 'borrowings', 'currentLiabilities', 'totalLiabilities', 'totalEquity', 'operatingCashFlow', 'capitalExpenditure', 'borrowingsRepaid'])
dict_keys(['profitability', 'liquidityAndCashFlow', 'operationalEfficiency', 'capitalStructure'])


### Check the latest year financial statement details.

In [25]:
statements = latest_year["statements"]

print("Revenue:", statements["revenue"])
print("Profit after tax:", statements["profitAfterTax"])
print("Cash:", statements["cash"])
print("Borrowings:", statements["borrowings"])
print("Total liabilities:", statements["totalLiabilities"])
print("Total equity:", statements["totalEquity"])
print("Operating cash flow:", statements["operatingCashFlow"])

Revenue: 5352856
Profit after tax: 96805
Cash: 528591
Borrowings: 2112710
Total liabilities: 3611401
Total equity: 4447731
Operating cash flow: 113617


In [26]:
def format_rm(value):
    return f"RM {value:,.0f}"

print("Revenue:", format_rm(statements["revenue"]))
print("Profit after tax:", format_rm(statements["profitAfterTax"]))

Revenue: RM 5,352,856
Profit after tax: RM 96,805


In [27]:
for year in sorted(
    financial_years,
    key=lambda item: item["financialYear"]
):
    statements = year["statements"]

    print(
        year["financialYear"],
        "Revenue:", format_rm(statements["revenue"]),
        "Profit after tax:", format_rm(statements["profitAfterTax"]),
        "Operating cash flow:", format_rm(statements["operatingCashFlow"]),
    )

2022 Revenue: RM 5,130,131 Profit after tax: RM 113,910 Operating cash flow: RM 127,469
2023 Revenue: RM 5,229,803 Profit after tax: RM 97,951 Operating cash flow: RM 228,345
2024 Revenue: RM 5,352,856 Profit after tax: RM 96,805 Operating cash flow: RM 113,617


Note, based on (Unity Global Trading Sdn. Bhd.):
- Is revenue increasing or decreasing? Increasing (Year by Year)
- Is the company profitable? Yes
- Is profit improving or worsening? Profit Decreasing even though revenue increase
- Is operating cash flow positive? Yes
- Does accounting profit agree with cash generation? Yes

Overall, while cash generation remains positive and supports reported profitability, the declining margins and volatility in operating cash flow warrant further investigation, particularly into working capital movements and operating costs.


### Ratios

In [28]:
ratios = latest_year["ratiosAndMetrics"]

for group_name, group_values in ratios.items():
    print("\n", group_name)
    pprint(group_values, sort_dicts=False)


 profitability
{'grossProfitMargin': 0.351,
 'operatingProfitMargin': 0.0455,
 'ebitda': 403948,
 'ebitdaMargin': 0.0755,
 'operatingExpenseRatio': 0.3055,
 'netProfitMargin': 0.0181,
 'taxEfficiencyRatio': 0.76}

 liquidityAndCashFlow
{'quickRatio': 1.3506,
 'cashRatio': 0.3221,
 'cashToRevenueRatio': 0.0987,
 'operatingCashFlow': 113617,
 'operatingCashFlowMargin': 0.0212,
 'freeCashFlow': -239509,
 'accrualRatio': -0.0021}

 operationalEfficiency
{'cashConversionCycle': 55.5105,
 'daysSalesOutstanding': 87.6067,
 'daysInventoryOutstanding': 22.075,
 'daysPayablesOutstanding': 54.1712,
 'inventoryTurnover': 16.5346,
 'receivablesTurnover': 4.1663,
 'assetTurnoverRatio': 0.6642,
 'netWorkingCapital': 785481}

 capitalStructure
{'leverageRatio': 0.812,
 'interestCoverageRatio': 2.0962,
 'fixedAssetComposition': 0.6989,
 'depreciationToAssetRatio': 0.0199,
 'debtServiceCoverageRatio': 1.0701}


In [29]:
# Take some example
net_margin = ratios["profitability"]["netProfitMargin"]
quick_ratio = ratios["liquidityAndCashFlow"]["quickRatio"]
leverage_ratio = ratios["capitalStructure"]["leverageRatio"]
debt_service_coverage = ratios["capitalStructure"]["debtServiceCoverageRatio"]

print("Net profit margin:", net_margin)
print("Quick ratio:", quick_ratio)
print("Leverage ratio:", leverage_ratio)
print("Debt-service coverage:", debt_service_coverage)

Net profit margin: 0.0181
Quick ratio: 1.3506
Leverage ratio: 0.812
Debt-service coverage: 1.0701


In [32]:
# margin is stored as a decimal like 0.12 = 12%, so

print(f"Net profit margin: {net_margin:.1%}")

# then for ratio,
print(f"Quick ratio: {quick_ratio:.2f}")

Net profit margin: 1.8%
Quick ratio: 1.35


### Check which companies dont have financial data

In [33]:
companies_without_financials = [
    company
    for company in companies
    if not company["financials"]["years"]
]

print("Companies without financial data:", len(companies_without_financials))

for company in companies_without_financials:
    print(
        company["registrationNumber"],
        company["profile"]["companyName"],
    )

Companies without financial data: 3
SYN-000302 Global Pacific Perunding Sdn. Bhd.
SYN-000333 Gemilang Harmoni Enterprise Sdn. Bhd.
SYN-001203 Pacific Seri Freight Sdn. Bhd.


Note:
This is an important edge case. 
The system should say that financial information is unavailable rather than inventing a financial assessment, for the 3 companies above !!

## Drill into Litigation part

Notes: 

Plaintiff - The company initiated the case, but this does not automatically mean the company is facing a liability.

Defendant - The claim was brought against the company. This may be more relevant as a potential risk, but it is still not proof that the company lost.

Claim amount - The claim amount is not necessarily:
- A confirmed loss
- A court judgment
- An amount already paid
- A recognised accounting liability

Case status - An ongoing case should not be described as a confirmed outcome.

In [34]:
company_with_litigation = next(
    company
    for company in companies
    if company["litigation"]["cases"]
)

print(company_with_litigation["profile"]["companyName"])
pprint(company_with_litigation["litigation"]["cases"][0], sort_dicts=False)

Teguh Kirana Freight Sdn. Bhd.
{'caseNumber': 'CASE-2024-7669',
 'role': 'plaintiff',
 'caseType': 'Debt recovery',
 'court': 'Sessions Court (Pulau Pinang)',
 'filedOn': '2024-09-26',
 'status': 'Settled',
 'claimAmount': 169800.0}


## Explore Profiles and Relationships

### Company Profiles

To Understand:
- The kinds of companies in the dataset
- Company status, industry and location
- Director and shareholder structures
- Ownership concentration
- Directors who are also shareholders
- Optional or missing relationship fields

In [35]:
selected_company = companies[0]
profile = selected_company["profile"]

pprint(profile, sort_dicts=False)

{'companyName': 'Unity Global Trading Sdn. Bhd.',
 'status': 'Active',
 'companyType': 'Private limited company',
 'incorporationDate': '2018-06-15',
 'industry': 'Retail',
 'state': 'Selangor',
 'natureOfBusiness': 'Retail of textiles and baju kurung',
 'shareCapital': 250000,
 'formerNames': []}


In [38]:
# Check the status of all 50 companies

from collections import Counter

status_counts = Counter(
    company["profile"]["status"]
    for company in companies
)

status_counts

Counter({'Active': 48, 'Winding up': 2})

In [40]:
# Count the industry from companies

industry_counts = Counter(
    company["profile"]["industry"]
    for company in companies
)

industry_counts.most_common()

[('Hospitality', 7),
 ('Food & Beverage', 7),
 ('Retail', 6),
 ('Healthcare', 5),
 ('Professional Services', 5),
 ('Technology', 5),
 ('Manufacturing', 4),
 ('Education', 3),
 ('Agriculture', 3),
 ('Construction', 3),
 ('Logistics', 2)]

Note: Different indutries might require different financial interpretations later, so dont assume the same standard across every industry !

In [42]:
# Count state of the comapnies

state_counts = Counter(
    company["profile"]["state"]
    for company in companies
)

state_counts.most_common()

[('Kuala Lumpur', 16),
 ('Selangor', 13),
 ('Pulau Pinang', 4),
 ('Sarawak', 3),
 ('Labuan', 3),
 ('Kelantan', 2),
 ('Sabah', 2),
 ('Perak', 2),
 ('Kedah', 2),
 ('Pahang', 1),
 ('Terengganu', 1),
 ('Perlis', 1)]

In [44]:
# Company Former Names
# former name is a factual identity detail, not automatically a warning sign

companies_with_former_names = [
    company
    for company in companies
    if company["profile"]["formerNames"]
]

print("Companies with former names:", len(companies_with_former_names))

for company in companies_with_former_names:
    print(
        company["registrationNumber"],
        company["profile"]["companyName"],
        company["profile"]["formerNames"],
    )

Companies with former names: 5
SYN-000267 Global Everbright Store Sdn. Bhd. [{'name': 'Sinar Industries Sdn. Bhd.', 'changedOn': '2025-06-25'}]
SYN-000333 Gemilang Harmoni Enterprise Sdn. Bhd. [{'name': 'Unity Industries Sdn. Bhd.', 'changedOn': '2025-09-03'}]
SYN-000451 Mutiara Delima Bakery Sdn. Bhd. [{'name': 'Gemilang Ventures Sdn. Bhd.', 'changedOn': '2025-11-27'}]
SYN-001199 Harmoni Murni Catering Sdn. Bhd. [{'name': 'Kirana Industries Sdn. Bhd.', 'changedOn': '2024-09-26'}]
SYN-001298 Mutiara Delima Agro Sdn. Bhd. [{'name': 'Sejahtera Resources Sdn. Bhd.', 'changedOn': '2023-11-27'}]


### Directors and Shareholders

In [45]:
# Inspect the relationships

relationships = selected_company["relationships"]

print("Directors:")
pprint(relationships["directors"], sort_dicts=False)

print("\nShareholders:")
pprint(relationships["shareholders"], sort_dicts=False)

Directors:
[{'name': 'Director 1',
  'designation': 'Managing Director',
  'appointedOn': '2021-02-05'},
 {'name': 'Director 2',
  'designation': 'Independent Director',
  'appointedOn': '2022-01-24'},
 {'name': 'Director 3', 'designation': 'Director', 'appointedOn': '2022-11-06'},
 {'name': 'Director 4', 'designation': 'Director', 'appointedOn': '2025-11-30'},
 {'name': 'Director 5',
  'designation': 'Executive Director',
  'appointedOn': '2025-11-01'}]

Shareholders:
[{'type': 'individual', 'name': 'Director 1', 'percentage': 12.21},
 {'type': 'individual', 'name': 'Director 5', 'percentage': 21.22},
 {'type': 'individual', 'name': 'Director 4', 'percentage': 58.25},
 {'type': 'individual', 'name': 'Director 3', 'percentage': 8.32}]


In [46]:
print("Number of directors:", len(relationships["directors"]))
print("Number of shareholders:", len(relationships["shareholders"]))

Number of directors: 5
Number of shareholders: 4


In [48]:
# Check ownership percentage

total_ownership = sum(
    shareholder["percentage"]
    for shareholder in relationships["shareholders"]
)

print(f"Total ownership: {total_ownership:.2f}%")

Total ownership: 100.00%


In [49]:
ownership_is_complete = abs(total_ownership - 100) < 0.01

print("Ownership totals approximately 100%:", ownership_is_complete)

Ownership totals approximately 100%: True


In [51]:
# Identify the largest shareholder

largest_shareholder = max(
    relationships["shareholders"],
    key=lambda shareholder: shareholder["percentage"]
)

print("Largest shareholder:", largest_shareholder["name"])
print(f"Ownership: {largest_shareholder['percentage']:.2f}%")
print("Type:", largest_shareholder["type"])

Largest shareholder: Director 4
Ownership: 58.25%
Type: individual


In [53]:
# Directors who are also shareholders

director_names = {
    director["name"]
    for director in relationships["directors"]
}

shareholder_names = {
    shareholder["name"]
    for shareholder in relationships["shareholders"]
}

director_shareholders = director_names & shareholder_names # find names in both sets

print("Directors who are also shareholders:", director_shareholders)

Directors who are also shareholders: {'Director 1', 'Director 3', 'Director 5', 'Director 4'}


In [54]:
# Inspect corporate shareholders

corporate_shareholders = [
    shareholder
    for company in companies
    for shareholder in company["relationships"]["shareholders"]
    if shareholder["type"] == "company"
]

print("Corporate shareholder records:", len(corporate_shareholders))

if corporate_shareholders:
    pprint(corporate_shareholders[0], sort_dicts=False)

Corporate shareholder records: 8
{'type': 'company',
 'name': 'Gemilang Holdings Sdn. Bhd.',
 'registrationNumber': 'SYN-009757',
 'incorporatedIn': 'Malaysia',
 'percentage': 51.0}


^ Belongs to Cahaya Harmoni Hotel Sdn. Bhd.

#### Profile observations

- The dataset contains companies from several industries and states.
- Most companies are active, while there are 2 are winding up.
- Some companies have former names, which may be useful identity context.

#### Relationship observations

- Some directors also hold shares in their companies.
- The dataset includes both individual and corporate shareholders.

In [57]:
ownership_checks = []

for company in companies:
    total = sum(
        shareholder["percentage"]
        for shareholder in company["relationships"]["shareholders"]
    )

    ownership_checks.append(
        {
            "registration_number": company["registrationNumber"],
            "total_ownership": total,
            "approximately_100": abs(total - 100) < 0.01,
        }
    )

invalid_ownership_totals = [
    check
    for check in ownership_checks
    if not check["approximately_100"]
]

invalid_ownership_totals

print("Companies checked:", len(ownership_checks))
print("Invalid totals:", len(invalid_ownership_totals))
print(
    "All totals approximately 100%:",
    len(invalid_ownership_totals) == 0,
)

Companies checked: 50
Invalid totals: 0
All totals approximately 100%: True


All 50 companies have shareholder percentages that total to approximately 100%, using a tolerance of 0.01 percentage points.

## Edge Cases and Development Sample

In [60]:
def get_latest_year(company):
    years = company["financials"]["years"]

    if not years:
        return None

    return max(
        years,
        key=lambda year: year["financialYear"],
    )

# Create simple company flags
def get_company_flags(company):
    flags = []

    profile = company["profile"]
    latest_year = get_latest_year(company)
    cases = company["litigation"]["cases"]
    shareholders = company["relationships"]["shareholders"]

    if profile["status"] != "Active":
        flags.append(f"status:{profile['status']}")

    if latest_year is None:
        flags.append("no_financial_data")
    else:
        statements = latest_year["statements"]

        if statements["profitAfterTax"] < 0:
            flags.append("latest_year_loss")

        if statements["operatingCashFlow"] < 0:
            flags.append("negative_operating_cash_flow")

        if latest_year["audit"]["opinion"] != "Unqualified":
            flags.append(
                f"audit:{latest_year['audit']['opinion']}"
            )

    if any(case["role"] == "defendant" for case in cases):
        flags.append("litigation_defendant")

    if any(case["role"] == "plaintiff" for case in cases):
        flags.append("litigation_plaintiff")

    if any(
        shareholder["type"] == "company"
        for shareholder in shareholders
    ):
        flags.append("corporate_shareholder")

    if any(
        shareholder["percentage"] >= 50
        for shareholder in shareholders
    ):
        flags.append("controlling_shareholder")

    return flags

In [62]:
# display flagged companies

for company in companies:
    flags = get_company_flags(company)

    if flags:
        print(
            company["registrationNumber"],
            "|",
            company["profile"]["companyName"],
            "|",
            ", ".join(flags),
        )

SYN-000029 | Unity Global Trading Sdn. Bhd. | controlling_shareholder
SYN-000067 | Jaya Prima Travel Sdn. Bhd. | latest_year_loss, negative_operating_cash_flow
SYN-000148 | Indah Prima Medical Sdn. Bhd. | latest_year_loss, negative_operating_cash_flow, controlling_shareholder
SYN-000165 | Teguh Kirana Freight Sdn. Bhd. | litigation_plaintiff, controlling_shareholder
SYN-000217 | Teguh Bestari Learning Sdn. Bhd. | controlling_shareholder
SYN-000267 | Global Everbright Store Sdn. Bhd. | litigation_plaintiff, controlling_shareholder
SYN-000282 | Delima Pacific Academy Sdn. Bhd. | controlling_shareholder
SYN-000286 | Cahaya Seri Catering Sdn. Bhd. | litigation_defendant, controlling_shareholder
SYN-000302 | Global Pacific Perunding Sdn. Bhd. | no_financial_data, controlling_shareholder
SYN-000333 | Gemilang Harmoni Enterprise Sdn. Bhd. | no_financial_data
SYN-000381 | Cahaya Harmoni Hotel Sdn. Bhd. | corporate_shareholder, controlling_shareholder
SYN-000397 | Pacific Pacific Education Sdn.

In [67]:
# Based on the flagged companies above, I chose 6 to further look into

development_ids = [
    "SYN-001102",
    "SYN-000217",
    "SYN-000148",
    "SYN-001203",
    "SYN-001070",
    "SYN-001212",
]

development_sample = [
    company
    for company in companies
    if company["registrationNumber"] in development_ids
]

for company in development_sample:
    print(
        company["registrationNumber"],
        company["profile"]["companyName"],
        get_company_flags(company),
    )

SYN-000148 Indah Prima Medical Sdn. Bhd. ['latest_year_loss', 'negative_operating_cash_flow', 'controlling_shareholder']
SYN-000217 Teguh Bestari Learning Sdn. Bhd. ['controlling_shareholder']
SYN-001070 Jaya Mega Consulting Sdn. Bhd. ['litigation_defendant', 'litigation_plaintiff', 'controlling_shareholder']
SYN-001102 Seri Maju Digital Sdn. Bhd. []
SYN-001203 Pacific Seri Freight Sdn. Bhd. ['no_financial_data', 'litigation_plaintiff', 'corporate_shareholder', 'controlling_shareholder']
SYN-001212 Sinar Sinar Pelancongan Sdn. Bhd. ['status:Winding up', 'litigation_defendant', 'litigation_plaintiff', 'controlling_shareholder']


#### Development sample rationale

- a small purposeful sample was selected for initial system development.
- The sample covers ordinary records as well as important edge cases, including missing financial data, winding-up status, litigation and weak financial performance.

- This sample is intended to expose different summarisation failure modes while limiting API usage during prompt development. It is not treated as statistically representative of all 50 companies.

- A separate set of companies should be used later to evaluate the system after prompt development, reducing the risk of evaluating only on records used during iteration.